## Example 1: Basic Markdown to HTML conversion

In [ ]:
import markdown

# Example 1: Basic Markdown to HTML conversion
print("Example 1: Basic Markdown to HTML Conversion")

assert markdown.markdown("# T") == "<h1>T</h1>"

# Sample Markdown content
markdown_text = """
# Welcome to Markdown Processing

This is a **comprehensive** example of Markdown processing capabilities.

## Features Demonstrated

- *Italic text* and **bold text**
- [Links to external sites](https://python.org)
- `Inline code` snippets

> This is a blockquote demonstrating
> multi-line quoted content.

    def hello_world():
        return 42
""".strip()

print(f"Original Markdown ({len(markdown_text)} characters)")

# Convert to HTML
md_processor = markdown.Markdown()
html_output = md_processor.convert(markdown_text)
print(f"\nConverted to HTML ({len(html_output)} characters):")
print(html_output)

expected_fragments = [
    "<h1>Welcome to Markdown Processing</h1>",
    "<p>This is a <strong>comprehensive</strong> example",
    "<h2>Features Demonstrated</h2>",
    "<li><em>Italic text</em> and <strong>bold text</strong></li>",
    '<li><a href="https://python.org">Links to external sites</a></li>',
    "<li><code>Inline code</code> snippets</li>",
    "<blockquote>\n<p>This is a blockquote demonstrating\nmulti-line quoted content.</p>\n</blockquote>",
    "<pre><code>def hello_world():\n    return 42\n</code></pre>",
]
for fragment in expected_fragments:
    assert fragment in html_output, (fragment, html_output)

# Processor details
print(f"\nProcessor details:")
# version string
print(f"  Markdown version: {markdown.__version__}")
print(f"  Processor type: {type(md_processor).__name__}")
print(f"  Tree processors: {[type(p).__name__ for p in md_processor.treeprocessors]}")
print(f"  Registered extensions: {md_processor.registeredExtensions}")
assert md_processor.registeredExtensions == []  # no extensions were requested

# Individual Markdown elements and the HTML each one produces
test_elements = {
    "Headers": ("# H1\n## H2\n### H3", "<h1>H1</h1>\n<h2>H2</h2>\n<h3>H3</h3>"),
    "Emphasis": ("*italic* and **bold** text", "<p><em>italic</em> and <strong>bold</strong> text</p>"),
    "Links": ("[Python](https://python.org)", '<p><a href="https://python.org">Python</a></p>'),
    "Lists": ("- Item 1\n- Item 2\n    - Nested item",
              "<ul>\n<li>Item 1</li>\n<li>Item 2<ul>\n<li>Nested item</li>\n</ul>\n</li>\n</ul>"),
    "Code": ("Inline `code` and\n\n    code block", "<p>Inline <code>code</code> and</p>\n<pre><code>code block\n</code></pre>"),
}

print(f"\nTesting individual elements:")
for element_name, (element_md, expected_html) in test_elements.items():
    element_html = md_processor.reset().convert(element_md)
    assert element_html == expected_html, (element_name, element_html)
    print(f"  {element_name}: {element_html!r}")

print("Basic Markdown conversion checked")

## Example 2: Extensions and advanced formatting

In [ ]:
import markdown

# Example 2: Extensions and advanced formatting
print("Example 2: Extensions and Advanced Formatting")

advanced_md = """
# Advanced Markdown Features

## Tables

| Feature | Status | Notes |
|---------|--------|-------|
| Headers | ✅ | Full support |
| Tables | ⚠️ | Extension required |

## Code with Language

```python
import markdown
html = markdown.markdown("**Hello World**")
```

## Definition Lists

Term 1
:   Definition for term 1

## Footnotes

This is text with footnotes[^1] and more footnotes[^2].

[^1]: This is the first footnote.
[^2]: This is the second footnote.
""".strip()

# Each extension adds the HTML construct it is responsible for; without it the source stays plain text
extension_markers = {
    "tables": "<table>",
    "fenced_code": '<pre><code class="language-python">',
    "def_list": "<dl>\n<dt>Term 1</dt>\n<dd>Definition for term 1</dd>\n</dl>",
    "footnotes": '<div class="footnote">',
    "toc": '<h2 id="tables">Tables</h2>',
}

plain_html = markdown.markdown(advanced_md)
print(f"No extensions: {len(plain_html)} characters of HTML")
for extension, marker in extension_markers.items():
    assert marker not in plain_html, (extension, marker)

for extension, marker in extension_markers.items():
    html_result = markdown.markdown(advanced_md, extensions=[extension])
    assert marker in html_result, (extension, html_result)
    print(f"  {extension}: produces {marker.splitlines()[0]!r}")

# All of them together
combined = markdown.markdown(advanced_md, extensions=list(extension_markers))
assert all(marker in combined for marker in extension_markers.values()), combined
assert combined.count('<sup id="fnref:') == 2, combined
assert "<td>Extension required</td>" in combined
print(f"\nAll {len(extension_markers)} extensions together: {len(combined)} characters of HTML")

print("Extensions and advanced formatting checked")

## Example 3: Custom configuration and output control

In [ ]:
import re
import markdown

# Example 3: Custom configuration and output control
print("Example 3: Custom Configuration and Output Control")

sample_md = """
# Document Processing Example

Line one  
line two after a hard break.

* * *

[External Link](http://example.com)

<a href="javascript:alert('xss')">Dangerous Link</a>

<script>alert('potentially dangerous')</script>
""".strip()

# output_format controls how void elements are closed
xhtml = markdown.markdown(sample_md, output_format="xhtml")
html5 = markdown.markdown(sample_md, output_format="html")
print(f"xhtml: {re.findall(r'<(?:br|hr)[^>]*>', xhtml)}")
print(f"html:  {re.findall(r'<(?:br|hr)[^>]*>', html5)}")
assert "<br />" in xhtml and "<hr />" in xhtml, xhtml
assert "<br>" in html5 and "<hr>" in html5 and "<br />" not in html5, html5

# tab_length sets how far a line must be indented to become a code block
indented = "Paragraph\n\n  indented two spaces"
assert "<pre><code>" not in markdown.markdown(indented)
assert "<pre><code>indented two spaces\n</code></pre>" in markdown.markdown(indented, tab_length=2)
print("tab_length=2 turns a two-space indent into a code block")

# Python-Markdown passes raw HTML through unchanged; sanitizing it is left to the caller
raw_html = markdown.markdown(sample_md)
assert "<script>alert('potentially dangerous')</script>" in raw_html, raw_html
assert "javascript:alert" in raw_html, raw_html
print("Raw HTML (including <script>) is passed through as-is")

def basic_html_sanitize(html_text):
    """Illustrative only: use a real sanitizer (for example nh3 or bleach) for untrusted input."""
    html_text = re.sub(r"<script[^>]*>.*?</script>", "", html_text, flags=re.DOTALL | re.IGNORECASE)
    return re.sub(r"javascript:[^\"']*", "#", html_text, flags=re.IGNORECASE)

sanitized_html = basic_html_sanitize(raw_html)
assert "<script" not in sanitized_html and "javascript:" not in sanitized_html, sanitized_html
assert '<a href="http://example.com">External Link</a>' in sanitized_html
print(f"After sanitizing: {len(raw_html)} -> {len(sanitized_html)} characters")

print("Custom configuration checked")

## Example 4: Document processing workflow and metadata

In [ ]:
import markdown

# Example 4: Document processing workflow and metadata
print("Example 4: Document Processing Workflow and Metadata")

# Markdown with YAML-style front matter, read by the `meta` extension
doc_with_metadata = """---
title: Technical Documentation
author: Development Team
version: 2.1
tags: python
      markdown
---

# Technical Documentation

This document demonstrates metadata handling in Markdown processing.

## Document Structure

The document includes front matter, structured content and cross-references.

### Implementation Details

See [Section 1](#document-structure) for more details.

## Conclusion

This demonstrates a complete document processing workflow.
"""

md_processor = markdown.Markdown(extensions=["meta", "toc"])
html_output = md_processor.convert(doc_with_metadata)
print(f"Document processed: {len(html_output)} characters HTML")

# meta: lower-cased keys, each value a list of lines
print(f"\nExtracted metadata: {md_processor.Meta}")
assert md_processor.Meta == {
    "title": ["Technical Documentation"],
    "author": ["Development Team"],
    "version": ["2.1"],
    "tags": ["python", "markdown"],
}, md_processor.Meta
assert "---" not in html_output and "title:" not in html_output

# toc: heading ids and a nested table of contents
outline = [(token["level"], token["name"], token["id"]) for token in md_processor.toc_tokens]
print(f"Top-level headings: {outline}")
assert outline == [(1, "Technical Documentation", "technical-documentation")], outline
children = [(t["level"], t["name"]) for t in md_processor.toc_tokens[0]["children"]]
assert children == [(2, "Document Structure"), (2, "Conclusion")], children
assert '<h2 id="document-structure">Document Structure</h2>' in html_output
assert '<a href="#document-structure">Section 1</a>' in html_output
assert '<div class="toc">' in md_processor.toc
print(f"Table of contents HTML: {len(md_processor.toc)} characters")

# reset() clears per-document state such as Meta before the next document
md_processor.reset()
md_processor.convert("# Second document")
assert md_processor.Meta == {}, md_processor.Meta

print("Document processing workflow checked")

## Example 5: Processor reuse and caching

In [ ]:
import markdown
import time

# Example 5: Processor reuse and caching
print("Example 5: Processor Reuse and Caching")

small_doc = "# Small Document\n\nThis is a **small** document for testing."
large_doc = "# Large Document\n\n" + "".join(
    "## Section {}\n\n{}\n\n".format(i, "Content paragraph. " * 20) for i in range(50)
)

for name, doc in {"small": small_doc, "large": large_doc}.items():
    start_time = time.perf_counter()
    html_result = markdown.markdown(doc)
    elapsed = time.perf_counter() - start_time
    print(f"{name}: {len(doc):,} -> {len(html_result):,} characters in {elapsed * 1000:.2f} ms")
assert markdown.markdown(large_doc).count("<h2>") == 50

# A processor can be reused for many documents: reset() it between conversions
reusable_processor = markdown.Markdown(extensions=["tables", "fenced_code"])
reused = [reusable_processor.reset().convert(small_doc) for _ in range(10)]
fresh = [markdown.Markdown(extensions=["tables", "fenced_code"]).convert(small_doc) for _ in range(10)]
assert reused == fresh, (reused[0], fresh[0])
assert reused[0] == "<h1>Small Document</h1>\n<p>This is a <strong>small</strong> document for testing.</p>"
print("Reused processor output matches a fresh processor for every document")


class MarkdownProcessor:
    """Reuses one processor and caches rendered HTML by source text."""

    def __init__(self, extensions=None, cache_size=100):
        self.processor = markdown.Markdown(extensions=extensions or ["tables", "toc"])
        self.cache = {}
        self.cache_size = cache_size
        self.hits = 0

    def process(self, md_content):
        if md_content in self.cache:
            self.hits += 1
            return self.cache[md_content]
        html_result = self.processor.reset().convert(md_content)
        if len(self.cache) < self.cache_size:
            self.cache[md_content] = html_result
        return html_result


prod_processor = MarkdownProcessor()
html1 = prod_processor.process(large_doc)
html2 = prod_processor.process(large_doc)
assert html1 is html2 and prod_processor.hits == 1 and len(prod_processor.cache) == 1
assert '<h2 id="section-49">Section 49</h2>' in html1  # toc adds heading ids
print(f"Cache: {len(prod_processor.cache)} entry, {prod_processor.hits} hit")

print("Processor reuse and caching checked")